# Statistical Comparative Analysis

**Purpose:** Rigorous statistical testing for comparing RL experiments

**Tests Included:**
- Paired t-test (parametric)
- Wilcoxon signed-rank test (non-parametric)
- Effect size (Cohen's d)
- Confidence intervals
- Normality testing (Shapiro-Wilk)
- Power analysis

**Usage:**
1. Load test results from JSON files
2. Run statistical comparisons
3. Generate publication-ready tables

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
from pathlib import Path

sns.set_style('whitegrid')
sns.set_context('notebook')

print("✅ Imports complete")

## 1. Load Experimental Results

In [ ]:
def load_test_results(results_path):
    """
    Load test results from JSON file.
    
    Args:
        results_path: Path to results JSON file
        
    Returns:
        Dictionary with scenarios and their errors
    """
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    results = {}
    for scenario_key, scenario_data in data.items():
        if 'errors' in scenario_data:
            results[scenario_key] = np.array(scenario_data['errors'])
    
    return results


# Load Phase 03 baseline
phase03_results = load_test_results(
    '../../../visualization_tools/plotting/phase03_test_results.json'
)

# Load Phase 04.2 results (when available)
# phase04_results = load_test_results(
#     '../../phase_04_state_optimization/04_2_trajectory_integration/session_data/test_results.json'
# )

print(f"✅ Loaded {len(phase03_results)} scenarios from Phase 03")
print(f"Scenarios: {list(phase03_results.keys())}")

## 2. Statistical Testing Functions

In [ ]:
def test_normality(data, alpha=0.05):
    """
    Test if data follows normal distribution using Shapiro-Wilk test.
    
    Args:
        data: Array of values
        alpha: Significance level
        
    Returns:
        Dictionary with test results
    """
    statistic, p_value = stats.shapiro(data)
    
    return {
        'statistic': statistic,
        'p_value': p_value,
        'is_normal': p_value > alpha,
        'interpretation': 'Normal' if p_value > alpha else 'Non-normal'
    }


def paired_t_test(data1, data2, alpha=0.05):
    """
    Perform paired t-test (parametric).
    
    Args:
        data1: First dataset
        data2: Second dataset
        alpha: Significance level
        
    Returns:
        Dictionary with test results
    """
    statistic, p_value = stats.ttest_rel(data1, data2)
    
    return {
        'test': 'Paired t-test',
        'statistic': statistic,
        'p_value': p_value,
        'significant': p_value < alpha,
        'interpretation': 'Significantly different' if p_value < alpha else 'Not significantly different'
    }


def wilcoxon_test(data1, data2, alpha=0.05):
    """
    Perform Wilcoxon signed-rank test (non-parametric).
    
    Args:
        data1: First dataset
        data2: Second dataset
        alpha: Significance level
        
    Returns:
        Dictionary with test results
    """
    statistic, p_value = stats.wilcoxon(data1, data2)
    
    return {
        'test': 'Wilcoxon signed-rank',
        'statistic': statistic,
        'p_value': p_value,
        'significant': p_value < alpha,
        'interpretation': 'Significantly different' if p_value < alpha else 'Not significantly different'
    }


def cohens_d(data1, data2):
    """
    Compute Cohen's d effect size.
    
    Interpretation:
    - |d| < 0.2: negligible
    - 0.2 ≤ |d| < 0.5: small
    - 0.5 ≤ |d| < 0.8: medium
    - |d| ≥ 0.8: large
    
    Args:
        data1: First dataset
        data2: Second dataset
        
    Returns:
        Dictionary with effect size results
    """
    mean_diff = np.mean(data1) - np.mean(data2)
    pooled_std = np.sqrt((np.std(data1, ddof=1)**2 + np.std(data2, ddof=1)**2) / 2)
    d = mean_diff / pooled_std
    
    # Interpretation
    abs_d = abs(d)
    if abs_d < 0.2:
        magnitude = 'Negligible'
    elif abs_d < 0.5:
        magnitude = 'Small'
    elif abs_d < 0.8:
        magnitude = 'Medium'
    else:
        magnitude = 'Large'
    
    return {
        'cohens_d': d,
        'magnitude': magnitude,
        'interpretation': f"{magnitude} effect size (d={d:.3f})"
    }


def confidence_interval(data, confidence=0.95):
    """
    Compute confidence interval for mean.
    
    Args:
        data: Array of values
        confidence: Confidence level (default: 0.95 for 95% CI)
        
    Returns:
        Dictionary with CI results
    """
    mean = np.mean(data)
    se = stats.sem(data)
    ci = stats.t.interval(confidence, len(data)-1, loc=mean, scale=se)
    
    return {
        'mean': mean,
        'ci_lower': ci[0],
        'ci_upper': ci[1],
        'ci_range': ci[1] - ci[0],
        'interpretation': f"{mean:.4f} [{ci[0]:.4f}, {ci[1]:.4f}]"
    }


print("✅ Statistical functions defined")

## 3. Comprehensive Comparison Function

In [ ]:
def compare_experiments(data1, data2, name1, name2, alpha=0.05):
    """
    Comprehensive statistical comparison between two experiments.
    
    Args:
        data1: First dataset
        data2: Second dataset
        name1: Name of first experiment
        name2: Name of second experiment
        alpha: Significance level
        
    Returns:
        Dictionary with all statistical results
    """
    results = {
        'experiments': f"{name1} vs {name2}",
        'n1': len(data1),
        'n2': len(data2),
    }
    
    # Descriptive statistics
    results['mean1'] = np.mean(data1)
    results['mean2'] = np.mean(data2)
    results['std1'] = np.std(data1, ddof=1)
    results['std2'] = np.std(data2, ddof=1)
    results['median1'] = np.median(data1)
    results['median2'] = np.median(data2)
    
    # Improvement
    results['mean_diff'] = results['mean1'] - results['mean2']
    results['percent_improvement'] = (results['mean1'] - results['mean2']) / results['mean1'] * 100
    
    # Normality tests
    results['normality1'] = test_normality(data1, alpha)
    results['normality2'] = test_normality(data2, alpha)
    
    # Choose appropriate test based on normality
    if results['normality1']['is_normal'] and results['normality2']['is_normal']:
        results['primary_test'] = paired_t_test(data1, data2, alpha)
        results['secondary_test'] = wilcoxon_test(data1, data2, alpha)  # Robustness check
    else:
        results['primary_test'] = wilcoxon_test(data1, data2, alpha)
        results['secondary_test'] = paired_t_test(data1, data2, alpha)  # For comparison
    
    # Effect size
    results['effect_size'] = cohens_d(data1, data2)
    
    # Confidence intervals
    results['ci1'] = confidence_interval(data1)
    results['ci2'] = confidence_interval(data2)
    
    return results


print("✅ Comparison function defined")

## 4. Run Statistical Comparisons

In [ ]:
# Example: Compare Phase 03 scenarios
comparison_results = {}

# Compare continuous vs impulse scenarios
if 'continuous_low' in phase03_results and 'impulse_low' in phase03_results:
    comp = compare_experiments(
        phase03_results['continuous_low'],
        phase03_results['impulse_low'],
        'Continuous Low',
        'Impulse Low'
    )
    comparison_results['continuous_vs_impulse_low'] = comp
    
    print("Continuous Low vs Impulse Low:")
    print(f"  Mean: {comp['mean1']:.4f} vs {comp['mean2']:.4f}")
    print(f"  Difference: {comp['mean_diff']:.4f} ({comp['percent_improvement']:.1f}%)")
    print(f"  p-value: {comp['primary_test']['p_value']:.4f}")
    print(f"  Effect size: {comp['effect_size']['interpretation']}")
    print(f"  Significant: {comp['primary_test']['significant']}")
    print()

# When Phase 04 data is available:
# comp_04 = compare_experiments(
#     phase03_results['continuous_low'],
#     phase04_results['continuous_low'],
#     'Phase 03',
#     'Phase 04.2'
# )
# comparison_results['phase03_vs_phase04'] = comp_04

print("✅ Comparisons complete")

## 5. Generate Publication-Ready Table

In [ ]:
def create_comparison_table(comparison_results):
    """
    Create publication-ready comparison table.
    
    Args:
        comparison_results: Dictionary of comparison results
        
    Returns:
        Pandas DataFrame
    """
    rows = []
    
    for key, comp in comparison_results.items():
        row = {
            'Comparison': comp['experiments'],
            'Mean 1': f"{comp['mean1']:.4f}",
            'Mean 2': f"{comp['mean2']:.4f}",
            'Δ': f"{comp['mean_diff']:.4f}",
            '% Improvement': f"{comp['percent_improvement']:.1f}%",
            'p-value': f"{comp['primary_test']['p_value']:.4f}",
            "Cohen's d": f"{comp['effect_size']['cohens_d']:.3f}",
            'Effect Size': comp['effect_size']['magnitude'],
            'Significant': '✓' if comp['primary_test']['significant'] else '✗',
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df


if comparison_results:
    table = create_comparison_table(comparison_results)
    print("\n📊 Comparison Table:")
    print(table.to_string(index=False))
    
    # Save to CSV
    output_path = "statistical_comparisons.csv"
    table.to_csv(output_path, index=False)
    print(f"\n✅ Table saved to: {output_path}")
else:
    print("⚠️ No comparisons to display yet")

## 6. Visualization: Box Plots with Significance

In [ ]:
def plot_comparison_boxplot(data1, data2, name1, name2, p_value):
    """
    Create box plot comparison with significance annotation.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Create box plots
    bp = ax.boxplot([data1, data2], labels=[name1, name2], patch_artist=True)
    
    # Color boxes
    colors = ['#3498db', '#e74c3c']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add significance annotation
    y_max = max(np.max(data1), np.max(data2))
    y_min = min(np.min(data1), np.min(data2))
    y_range = y_max - y_min
    
    # Significance stars
    if p_value < 0.001:
        sig_text = '***'
    elif p_value < 0.01:
        sig_text = '**'
    elif p_value < 0.05:
        sig_text = '*'
    else:
        sig_text = 'ns'
    
    # Draw significance line
    y = y_max + y_range * 0.05
    ax.plot([1, 2], [y, y], 'k-', lw=1.5)
    ax.text(1.5, y, sig_text, ha='center', va='bottom', fontsize=14)
    
    ax.set_ylabel('Position Error (m)', fontsize=12)
    ax.set_title(f'{name1} vs {name2}\np = {p_value:.4f}', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig


# Example plot
if 'continuous_vs_impulse_low' in comparison_results:
    comp = comparison_results['continuous_vs_impulse_low']
    fig = plot_comparison_boxplot(
        phase03_results['continuous_low'],
        phase03_results['impulse_low'],
        'Continuous Low',
        'Impulse Low',
        comp['primary_test']['p_value']
    )
    plt.show()
    plt.close()
    print("✅ Comparison plot generated")
else:
    print("⚠️ No data for visualization yet")

## 7. Template for Phase 03 vs Phase 04 Comparison

**Use this when Phase 04 data becomes available:**

In [ ]:
# Uncomment when Phase 04.2 results are available

# # Load Phase 04.2 results
# phase04_results = load_test_results(
#     '../../phase_04_state_optimization/04_2_trajectory_integration/session_data/test_results.json'
# )

# # Compare all matching scenarios
# phase_comparisons = {}
# scenarios = ['continuous_low', 'continuous_medium', 'continuous_high',
#              'impulse_low', 'impulse_medium', 'impulse_high']

# for scenario in scenarios:
#     if scenario in phase03_results and scenario in phase04_results:
#         comp = compare_experiments(
#             phase03_results[scenario],
#             phase04_results[scenario],
#             f'Phase 03 - {scenario}',
#             f'Phase 04.2 - {scenario}'
#         )
#         phase_comparisons[scenario] = comp

# # Generate comprehensive table
# comparison_table = create_comparison_table(phase_comparisons)
# print(comparison_table)

# # Save to file
# comparison_table.to_csv('phase03_vs_phase04_statistical_comparison.csv', index=False)
# comparison_table.to_latex('phase03_vs_phase04_statistical_comparison.tex', index=False)

# print("\n✅ Phase 03 vs Phase 04 comparison complete!")
# print("📄 Results saved to CSV and LaTeX formats")

print("⏳ Waiting for Phase 04 data...")

## Summary

**This notebook provides:**

✅ Normality testing (Shapiro-Wilk)  
✅ Parametric tests (paired t-test)  
✅ Non-parametric tests (Wilcoxon)  
✅ Effect size calculation (Cohen's d)  
✅ Confidence intervals (95% CI)  
✅ Publication-ready tables  
✅ Statistical significance visualization  

**Next Steps:**
1. Run Phase 04.2 experiments
2. Load results into this notebook
3. Generate comparison tables
4. Include in research paper/report